# 02 — Feature economics and decay

Three experiments against the grouped holdout from Stage 1:

1. **Tier ablation** — URL-string only vs +HTML vs +TLS/WHOIS vs all 30.
2. **Decay simulation** — force `SSLfinal_State = 1` (universal HTTPS) and neutralize Alexa/PageRank.
3. **Adversarial flips** — an attacker sets the *k* cheapest features to legitimate on phishing rows.

The live scanner loads a **25-feature** model that was never trained on the five 2026-unobtainable reputation features, rather than feeding a 30-feature model constant placeholders.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from phishing.config import REPORTS_DIR, UNAVAILABLE_2026
from phishing.data import grouped_split, load_xy
from phishing.decay import adversarial_curve, decay_simulation, tier_ablation
from phishing.models import build_model

pd.set_option("display.float_format", "{:.4f}".format)
print("unobtainable in 2026:", UNAVAILABLE_2026)

In [ ]:
def _load_or(name, fn):
    path = REPORTS_DIR / name
    if path.exists():
        return pd.read_csv(path, index_col=0)
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    df = fn()
    df.to_csv(path)
    return df

X, y, groups = load_xy()
X_tr, X_te, y_tr, y_te, _, _ = grouped_split(X, y, groups)

tiers = _load_or("tier_ablation.csv", lambda: tier_ablation(X_tr, y_tr, X_te, y_te))
print("=== tier ablation ===")
display(tiers[["n_features", "accuracy", "auroc", "f1", "recall"]])

In [ ]:
rf = build_model("Random Forest")
decay = _load_or("decay_simulation.csv", lambda: decay_simulation(rf, X_tr, y_tr, X_te, y_te))
print("=== decay simulation ===")
display(decay[["accuracy", "auroc", "f1", "recall"]])

adv = _load_or("adversarial_curve.csv", lambda: adversarial_curve(rf, X_tr, y_tr, X_te, y_te, max_k=10))
print("=== adversarial cheapest-k flips (recall on mixed test set) ===")
display(adv[["accuracy", "recall", "f1", "flipped"]])

`SSLfinal_State` dominated the original Random Forest (32% of Gini importance) and Gradient Boosting (71%). Setting it to legitimate for every test row is the 2026 HTTPS world. The drop relative to `original_2012` is the amount of the 2012 model that was really "does this site have a serious certificate?".

SHAP global importance for the fitted forest:

In [ ]:
from phishing.explain import global_importance, shap_values

rf.fit(X_tr, y_tr)
sample = X_te.sample(n=min(400, len(X_te)), random_state=42)
_, values = shap_values(rf, sample, background=X_tr.sample(200, random_state=42))
global_importance(list(X_tr.columns), values).head(10)